# 01_ocel — Plugins and the OCEL event log

A plugin is an **observer, not a participant**: at each step boundary the machine emits a typed event, the plugin receives it — and cannot change `params`, `state`, or `result`. Business code contains no observability lines.

`OcelPlugin` records each run in **OCEL 2.0** (Object-Centric Event Log) format for process mining. An aspect that wants to participate returns `OcelFrame` rows under `OCEL_FRAMES_KEY`; the store writes a JSON log on `close()`.

**What's new**

| Concept | Description |
|---------|-------------|
| `ActionProductMachine(plugins=[...])` | Plugins are wired as a list; business code unchanged |
| `OcelPlugin(store=, short_names=)` | Writes runs to an OCEL 2.0 log |
| `OcelFrame(object=entity, qualifier=...)` under `OCEL_FRAMES_KEY` | What an aspect contributes to the log |
| `InMemoryOcelStoreResource(output_file=)` | Holds events; writes JSON on `close()` |
| observer-not-participant | the plugin reads everything, changes nothing |

> Install with the OCEL extra: `pip install "aoa-action-machine[ocel]"`.

In [ ]:
!pip install "aoa-action-machine[ocel]"

In [ ]:
import asyncio
import json
from pathlib import Path

from pydantic import Field

from aoa.action_machine.auth import NoneRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.domain.entity import BaseEntity
from aoa.action_machine.intents.aspects import regular_aspect, summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.checkers import result_instance, result_string
from aoa.action_machine.intents.entity import entity
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.logging import Channel, ConsoleLogger
from aoa.action_machine.logging.log_coordinator import LogCoordinator
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.plugin.ocel import OCEL_FRAMES_KEY, InMemoryOcelStoreResource, OcelFrame, OcelPlugin
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

## Domain, Entity, Params, Result

In [ ]:
class RootDomain(BaseDomain):
    name = "root"
    description = "Root domain"


@entity(description="Order", domain=RootDomain)
class OrderEntity(BaseEntity):
    id: str = Field(description="Order ID")


class OrderParams(BaseParams):
    order_id: str = Field(description="Order ID")


class OrderResult(BaseResult):
    order_id: str = Field(description="Created order ID")

## The action — contribute OCEL frames

`validate_aspect` returns an `OcelFrame` (an `OrderEntity` object with a qualifier) under `OCEL_FRAMES_KEY`. The `@result_instance(OCEL_FRAMES_KEY, list, required=False)` checker keeps the frames list typed and optional.

In [ ]:
@meta(description="Create order", domain=RootDomain)
@check_roles(NoneRole)
class CreateOrderAction(BaseAction[OrderParams, OrderResult]):

    @regular_aspect("Validation")
    @result_string("validated_id", required=True)
    @result_instance(OCEL_FRAMES_KEY, list, required=False)
    async def validate_aspect(self, params, state, box, connections):
        await box.info(
            Channel.business,
            "Validate order: id={%var.order_id}",
            order_id=params.order_id,
        )
        order = OrderEntity(id=params.order_id)
        return {
            "validated_id": params.order_id,
            OCEL_FRAMES_KEY: [
                OcelFrame(object=order, qualifier="Create order"),
            ],
        }

    @summary_aspect("Create")
    async def create_summary(self, params, state, box, connections):
        return OrderResult(order_id=str(state["validated_id"]))

## Run

Three runs of `CreateOrderAction` produce three events in `ocel_log.json` — without a single logging line in the business logic. Such a log opens in process-mining tools (e.g. OC-PM): process graphs, funnels, conformance checks.

> In Colab, `await` works at top level — no `asyncio.run()`.

In [ ]:
async def main() -> None:
    output = Path("ocel_log.json")
    store = InMemoryOcelStoreResource(output_file=output)

    machine = ActionProductMachine(
        log_coordinator=LogCoordinator(loggers=[ConsoleLogger()]),
        plugins=[OcelPlugin(store=store, short_names=True)],
    )

    await store.open()
    for order_id in ["ord-001", "ord-002", "ord-003"]:
        await machine.run(
            Context(),
            CreateOrderAction(),
            params=OrderParams(order_id=order_id),
        )

    await store.close()
    data = json.loads(output.read_text())
    print("\n" + f"written: {output}")
    print(f"events: {len(data.get('events', []))}")


await main()